# ECEi MC evaluation dataloader

Isolated dataloader for **ecei_mc** decimated H5 data, matching the setup used by
`run_tcn_baseline_160_original_instancenorm.sh` for training. Use this notebook to
evaluate TCN checkpoints on the same data (clear_decimated + disrupt_decimated).

**Data paths (SciServer):**
- Disrupt: `/home/idies/workspace/Storage/yhuang2/persistent/ecei_mc/disrupt_decimated`
- Clear:   `/home/idies/workspace/Storage/yhuang2/persistent/ecei_mc/clear_decimated`

## Setup paths and imports

In [ ]:
import sys
from pathlib import Path

# soen_fusion_zero root: prefer cwd if we're in it, else parent, else fallback
ROOT = Path.cwd()
if (ROOT / "train_tcn_ddp_original.py").exists():
    pass
elif (ROOT.parent / "train_tcn_ddp_original.py").exists():
    ROOT = ROOT.parent
else:
    ROOT = Path("/home/idies/workspace/Storage/yhuang2/persistent/soen_fusion_zero")
sys.path.insert(0, str(ROOT))

# ecei_mc decimated H5 folders (same as run_tcn_baseline_160_original_instancenorm.sh)
DECIMATED_ROOT = "/home/idies/workspace/Storage/yhuang2/persistent/ecei_mc/disrupt_decimated"
CLEAR_DECIMATED_ROOT = "/home/idies/workspace/Storage/yhuang2/persistent/ecei_mc/clear_decimated"
NORM_STATS = "/home/idies/workspace/Storage/yhuang2/persistent/ecei/norm_stats.npz"
DISRUPT_FILE = str(ROOT / "disruptcnn" / "shots" / "d3d_disrupt_ecei.final.txt")
CLEAR_FILE = str(ROOT / "disruptcnn" / "shots" / "d3d_clear_ecei.final.txt")

print("ROOT:", ROOT)
print("DECIMATED_ROOT:", DECIMATED_ROOT)
print("CLEAR_DECIMATED_ROOT:", CLEAR_DECIMATED_ROOT)

In [ ]:
import numpy as np
import torch
from torch.utils.data import DataLoader, Subset

from disruptcnn.dataset_original import EceiDatasetOriginal, OriginalStyleDatasetForDDP

## Build dataset (same config as 160-original-instancenorm training)

Parameters match `train_tcn_ddp_original.py` defaults used by the run script: flattop_only, data_step=10, nsub=781_250, nrecept from 30k output receptive field.

In [ ]:
DATA_STEP = 10
NSUB = 781_250
NRECEPT_TARGET = 30_000  # output receptive field; dataset uses nrecept_raw = NRECEPT_TARGET * DATA_STEP
NRECEPT_RAW = NRECEPT_TARGET * DATA_STEP

inner_ds = EceiDatasetOriginal(
    root="/home/idies/workspace/Storage/yhuang2/persistent/ecei/dsrpt",  # not used when decimated_root set
    disrupt_file=DISRUPT_FILE,
    clear_file=CLEAR_FILE,
    flattop_only=True,
    normalize=True,
    data_step=DATA_STEP,
    nsub=NSUB,
    nrecept=NRECEPT_RAW,
    decimated_root=DECIMATED_ROOT,
    clear_decimated_root=CLEAR_DECIMATED_ROOT,
    norm_stats_path=NORM_STATS,
    decimate_extra=None,
)
ds = OriginalStyleDatasetForDDP(inner_ds)

print("Total sequences:", len(ds))
print("seq_has_disrupt sum:", ds.seq_has_disrupt.sum())

## Train/val/test splits and eval dataloader

In [ ]:
train_idx = ds.get_split_indices("train")
val_idx = ds.get_split_indices("test")
if len(val_idx) == 0:
    val_idx = ds.get_split_indices("val")

print("Train:", len(train_idx), "Val:", len(val_idx))

val_subset = Subset(ds, val_idx)
eval_loader = DataLoader(
    val_subset,
    batch_size=32,
    shuffle=False,
    num_workers=0,
    pin_memory=False,
    drop_last=False,
)

# One batch for shape check
X, target, weight = next(iter(eval_loader))
print("Batch X shape:", X.shape, "target:", target.shape)

## Visualize one test shot: original (flattop) vs windowed reconstruction

Pick one shot from the val/test set, load the full flattop segment from H5, and reconstruct the same segment from the model's windowed subsequences (overlap-averaged). Side-by-side comparison.

In [ ]:
import matplotlib.pyplot as plt
import h5py

# Pick one shot that has at least one sequence in val set
inner = ds._inner
shot_idxi = inner.shot_idxi  # per-seq: which shot (index) each seq belongs to
val_shots = np.unique(shot_idxi[val_idx])
if len(val_shots) == 0:
    raise RuntimeError("No val indices; run the split cell first.")
# Prefer a disruptive shot for clearer visualization
chosen_shot_idx = val_shots[0]
for s in val_shots:
    if inner.disrupted[s]:
        chosen_shot_idx = s
        break
seq_inds = val_idx[shot_idxi[val_idx] == chosen_shot_idx]
shot_number = int(inner.shot[chosen_shot_idx])
is_disrupt = bool(inner.disrupted[chosen_shot_idx])
print(f"Shot {shot_number} (disrupt={is_disrupt}), {len(seq_inds)} windows in val set")

# 1) Original full flattop segment: read from H5 (same norm/offset as dataset)
s = chosen_shot_idx
start, stop = int(inner.start_idx[s]), int(inner.stop_idx[s])
step = getattr(inner, "_step_in_getitem", 1)
filename = inner._filename(s)
with h5py.File(filename, "r") as f:
    if np.all(inner.offsets[..., s] == 0) and "offsets" in f:
        inner.offsets[..., s] = f["offsets"][...]
    LFS = np.asarray(f["LFS"][..., start:stop][..., ::step], dtype=np.float64)
off = inner.offsets[..., s]
if off.ndim >= 1 and off.shape[-1] > 1:
    t_len = off.shape[-1]
    s2 = min(stop, t_len)
    s1 = min(start, s2)
    off_slice = off[..., s1:s2][..., ::step]
    if off_slice.shape != LFS.shape:
        off_slice = np.zeros_like(LFS)
else:
    off_slice = off
orig = LFS - off_slice
if inner.normalize:
    orig = (orig - inner.normalize_mean[..., np.newaxis]) / inner.normalize_std[..., np.newaxis]
if orig.ndim == 2 and orig.shape[-1] < orig.shape[0]:
    orig = orig.T
# orig: (C, T_full)
T_full = orig.shape[-1]
disrupt_sample = None
if is_disrupt and inner.disrupt_idx[s] >= 0:
    # First output index where label=1 (same formula as __getitem__)
    disrupt_sample = int((inner.disrupt_idx[s] - start + 1) // step)
    disrupt_sample = max(0, min(disrupt_sample, T_full - 1))

# 2) Reconstruction from windowed data: overlap-average windows into full length
recon = np.zeros((*orig.shape[:-1], T_full), dtype=np.float64)
count = np.zeros(T_full, dtype=np.float64)
T_fixed = ds._T_fixed
for idx in seq_inds:
    X, _, _ = ds[idx]
    X = X.numpy() if isinstance(X, torch.Tensor) else np.asarray(X)
    # Window in output (decimated) space
    win_len = inner.stop_idxi[idx] - inner.start_idxi[idx] + 1
    win_len_out = win_len // step
    t0 = inner.start_idxi[idx] - start
    t0_out = t0 // step
    n = min(win_len_out, T_fixed, T_full - t0_out)
    if n <= 0:
        continue
    recon[..., t0_out : t0_out + n] += X[..., :n]
    count[t0_out : t0_out + n] += 1
count = np.maximum(count, 1e-10)
recon = recon / count[np.newaxis, :]

# Plot: mean over channels (or single channel)
if orig.shape[0] > 1:
    orig_plot = np.mean(orig, axis=0)
    recon_plot = np.mean(recon, axis=0)
else:
    orig_plot = orig[0]
    recon_plot = recon[0]
x = np.arange(T_full)

fig, (ax0, ax1) = plt.subplots(1, 2, figsize=(12, 3.5), sharey=True)
ax0.plot(x, orig_plot, color="C0", linewidth=0.6, label="Original (flattop)")
if disrupt_sample is not None:
    ax0.axvline(disrupt_sample, color="red", linestyle="--", alpha=0.8, label="t_disrupt")
ax0.set_title(f"Shot {shot_number} — original segment")
ax0.set_xlabel("Time (decimated samples)")
ax0.legend(loc="upper right")
ax0.grid(True, alpha=0.3)

ax1.plot(x, recon_plot, color="C1", linewidth=0.6, label="From windows (overlap-avg)")
if disrupt_sample is not None:
    ax1.axvline(disrupt_sample, color="red", linestyle="--", alpha=0.8, label="t_disrupt")
ax1.set_title(f"Shot {shot_number} — reconstruction from {len(seq_inds)} windows")
ax1.set_xlabel("Time (decimated samples)")
ax1.legend(loc="upper right")
ax1.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Optional: load checkpoint and run evaluation

Set `CHECKPOINT_PATH` to a `best.pt` or `epoch_XXXX.pt` from training. The model is built with the same TCN config (levels, nhid, kernel_size, etc.) and instance norm as in the run script.

In [ ]:
CHECKPOINT_PATH = None  # e.g. ROOT / "checkpoints_tcn_ddp_original/L4_H80_YYYYMMDD_HHMMSS/best.pt"

if CHECKPOINT_PATH and Path(CHECKPOINT_PATH).exists():
    from train_tcn_ddp_original import build_model
    ckpt = torch.load(CHECKPOINT_PATH, map_location="cpu", weights_only=False)
    args = ckpt.get("args", {})
    nrecept_ckpt = ckpt.get("nrecept", NRECEPT_TARGET)
    model, nrecept, _ = build_model(
        args.get("input_channels", 160), 1,
        args.get("levels", 4), args.get("nhid", 80),
        args.get("kernel_size", 15), args.get("dilation_base", 10), args.get("dropout", 0.1),
        nrecept_target=nrecept_ckpt,
        use_instance_norm=args.get("use_instance_norm", True),
        use_prenorm=args.get("use_prenorm", False),
    )
    state = ckpt["state_dict"]
    if next(iter(state.keys())).startswith("module."):
        state = {k.replace("module.", ""): v for k, v in state.items()}
    model.load_state_dict(state, strict=True)
    model.eval()
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = model.to(device)
    print("Model loaded. nrecept =", nrecept)
else:
    model = None
    nrecept = NRECEPT_TARGET
    print("Set CHECKPOINT_PATH to run model evaluation.")